# Stage 14 v1 — FoR robustness run

This notebook runs the reviewed Stage 14 v1 runner for exactly two separate model sets: `baseline-v1` and the frozen F4 bundle seed. It reuses Stage 11 clean scores and never makes another clean pass.

The default `MODE = 'check'` performs every runtime, artifact, clean-score and D58 check without opening FoR test audio or writing a registry claim. Change it to `official` only after the exact commit and this notebook are approved. A failed or interrupted official claim is not retried without a new decision.

In [ ]:
REPO_URL = 'https://github.com/kahlanzzi/voice-spoofing-detection.git'
COMMIT = 'REPLACE_WITH_APPROVED_COMMIT'  # full 40-character SHA
MODE = 'check'  # 'check' or 'official'
COLAB_RUNTIME_VERSION = '2026.10'
DRIVE_ROOT = '/content/drive/MyDrive/voice-spoofing-detection'
RESULTS_REF = 'docs/results/stage14_v1.md'
assert MODE in ('check', 'official'), MODE


## 1. Mount Drive and define the frozen inputs

Select **T4 GPU** and runtime version **2026.10** before running. The live append-only registry and Stage 11 clean per-clip files stay on Drive. Official Stage 14 outputs use new model-specific folders and are never overwritten.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tarfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path(DRIVE_ROOT)
STAGE11 = DRIVE / 'processed' / 'metadata' / 'stage11'
STAGE14 = DRIVE / 'processed' / 'metadata' / 'stage14' / 'v1'
REGISTRY = STAGE11 / 'run_registry.jsonl'
CLEAN_SCORES = STAGE11 / 'run' / 'runner' / 'clip_scores'
RUN_DEST = STAGE14 / 'run'
BASELINE_OUT = RUN_DEST / 'baseline-v1'
BUNDLE_OUT = RUN_DEST / 'bundle-v1'

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

assert REGISTRY.is_file(), REGISTRY
assert CLEAN_SCORES.is_dir(), CLEAN_SCORES
if MODE == 'official':
    assert not RUN_DEST.exists(), f'{RUN_DEST} already exists; outputs are never overwritten'
print('Mode:', MODE)


## 2. Check out the exact reviewed commit


In [ ]:
REPO = Path('/content/voice-spoofing-detection')
assert len(COMMIT) == 40 and all(c in '0123456789abcdef' for c in COMMIT), COMMIT
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=REPO, check=True)
HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
STATUS = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True)
assert HEAD == COMMIT and STATUS == '', (HEAD, STATUS)
os.chdir(REPO)
print('Exact clean commit:', HEAD)


## 3. Assert the D58 runtime and run the reviewed tests


In [ ]:
import platform, torch
GPU = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
assert COLAB_RUNTIME_VERSION == '2026.10'
assert platform.python_version() == '3.13.16', platform.python_version()
assert torch.__version__ == '2.11.0+cu130', torch.__version__
assert torch.cuda.is_available() and 'T4' in GPU, GPU
env = {**os.environ, 'REQUIRE_TORCH_TESTS': '1'}
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/evaluation', 'tests/governance'], cwd=REPO, env=env, check=True)
assert subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True) == ''
print('Runtime and tests passed:', GPU)


## 4. Verify and extract the pinned FoR archive


In [ ]:
import yaml
audit = yaml.safe_load((REPO / 'configs/audit.yaml').read_text(encoding='utf-8'))['for2sec']
ARCHIVE = DRIVE / 'raw' / 'for_2sec_ff8c82c7.tar'
REVISION = DRIVE / 'raw' / 'for_2sec_REVISION.txt'
assert REVISION.read_text(encoding='utf-8').strip() == audit['revision']
DATA_ROOT = Path('/content/data')
AUDIO_ROOT = DATA_ROOT / 'for_2sec'
LOCAL_ARCHIVE = DATA_ROOT / ARCHIVE.name
DATA_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
if AUDIO_ROOT.exists():
    shutil.rmtree(AUDIO_ROOT)
with tarfile.open(LOCAL_ARCHIVE) as source:
    source.extractall(DATA_ROOT, filter='data')
print('FoR archive SHA-256:', sha256(LOCAL_ARCHIVE))


## 5. Build the D58 val-A caches and copy frozen artifacts outside the clone


In [ ]:
MANIFEST = REPO / 'data/manifests/for2sec_frozen_manifest.csv'
PROFILE = REPO / 'configs/preprocessing.yaml'
CACHE_A, CACHE_B = Path('/content/stage14_cache_a'), Path('/content/stage14_cache_b')
for cache in (CACHE_A, CACHE_B):
    if cache.exists():
        shutil.rmtree(cache)
    for feature, module in {'log_mel': 'src.features.logmel_cache', 'mfcc': 'src.features.mfcc_cache'}.items():
        subprocess.run([sys.executable, '-m', module, '--root', str(AUDIO_ROOT), '--manifest', str(MANIFEST), '--profile', str(PROFILE), '--split', 'val-A', '--out', str(cache / feature / 'val-A')], cwd=REPO, check=True)

ARTIFACTS = Path('/content/stage14_artifacts')
if ARTIFACTS.exists():
    shutil.rmtree(ARTIFACTS)
BASELINE = ARTIFACTS / 'baseline-v1'
F4_BUNDLE = ARTIFACTS / 'bundle-v1'
MFCC_REFERENCE = ARTIFACTS / 'mfcc_reference'
shutil.copytree(DRIVE / 'checkpoints' / 'baseline' / 'baseline-v1', BASELINE)
shutil.copytree(DRIVE / 'checkpoints' / 'bundle' / 'bundle-v1', F4_BUNDLE)
shutil.copytree(STAGE11 / 'mfcc_reference', MFCC_REFERENCE)
assert not ARTIFACTS.resolve().is_relative_to(REPO.resolve())
print('Frozen artifacts copied outside the repository')


## 6. Read-only preflight for both model sets

This is the safe stopping point in `check` mode. It does not create a registry record or read a test clip.

In [ ]:
RUNNER = 'src.evaluation.stage14_run'
STAGE11_RUN_REPORT = STAGE11 / 'run' / 'runner' / 'run_report.json'
assert STAGE11_RUN_REPORT.is_file(), STAGE11_RUN_REPORT
COMMON = [
    '--registry', str(REGISTRY), '--clean-scores', str(CLEAN_SCORES),
    '--stage11-run-report', str(STAGE11_RUN_REPORT),
    '--profile', str(PROFILE), '--manifest', str(MANIFEST),
    '--robustness-config', str(REPO / 'configs/robustness.yaml'),
    '--augmentation-config', str(REPO / 'configs/augmentation.yaml'),
    '--colab-runtime-version', COLAB_RUNTIME_VERSION,
    '--stage7-evidence', str(REPO / 'docs/audit_evidence/stage7_baseline'),
    '--stage8-evidence', str(REPO / 'docs/audit_evidence/stage8_cnn'),
    '--stage10-evidence', str(REPO / 'docs/audit_evidence/stage10_risk'),
    '--baseline', str(BASELINE), '--f4-bundle', str(F4_BUNDLE),
    '--f4-record', str(REPO / 'docs/audit_evidence/stage12_bundle/f4_record.json'),
    '--cache-a', str(CACHE_A), '--cache-b', str(CACHE_B),
    '--mfcc-reference', str(MFCC_REFERENCE),
]
for kind in ('baseline', 'cnn_bundle'):
    subprocess.run([sys.executable, '-m', RUNNER, 'check', '--model-kind', kind, *COMMON], cwd=REPO, check=True)
assert subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True) == ''
print('Both read-only preflights passed. In check mode, stop here.')


## 7. Official baseline claim, durable remount, and score


In [ ]:
assert MODE == 'official', 'check mode: stop before the first claim'
subprocess.run([sys.executable, '-m', RUNNER, 'claim', '--model-kind', 'baseline', *COMMON, '--results-ref', RESULTS_REF, '--out', str(BASELINE_OUT)], cwd=REPO, check=True)
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
BASELINE_CLAIM = json.loads((BASELINE_OUT / 'claim.json').read_text(encoding='utf-8'))
assert sha256(REGISTRY) == BASELINE_CLAIM['registry_sha256']
subprocess.run([sys.executable, '-m', RUNNER, 'score', '--model-kind', 'baseline', *COMMON, '--root', str(AUDIO_ROOT), '--run-id', BASELINE_CLAIM['run_id'], '--out', str(BASELINE_OUT)], cwd=REPO, check=True)
print('baseline-v1 completed:', BASELINE_CLAIM['run_id'])


## 8. Official F4 bundle claim, durable remount, and score


In [ ]:
assert MODE == 'official'
subprocess.run([sys.executable, '-m', RUNNER, 'claim', '--model-kind', 'cnn_bundle', *COMMON, '--results-ref', RESULTS_REF, '--out', str(BUNDLE_OUT)], cwd=REPO, check=True)
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
BUNDLE_CLAIM = json.loads((BUNDLE_OUT / 'claim.json').read_text(encoding='utf-8'))
assert sha256(REGISTRY) == BUNDLE_CLAIM['registry_sha256']
subprocess.run([sys.executable, '-m', RUNNER, 'score', '--model-kind', 'cnn_bundle', *COMMON, '--root', str(AUDIO_ROOT), '--run-id', BUNDLE_CLAIM['run_id'], '--out', str(BUNDLE_OUT)], cwd=REPO, check=True)
print('F4 bundle seed completed:', BUNDLE_CLAIM['run_id'])


## 9. Verify the final Drive records and print the review summary


In [ ]:
assert MODE == 'official'
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
for folder in (BASELINE_OUT, BUNDLE_OUT):
    report = json.loads((folder / 'run_report.json').read_text(encoding='utf-8'))
    metrics = json.loads((folder / 'metrics_report.json').read_text(encoding='utf-8'))
    assert report['code_commit'] == COMMIT
    assert len(report['conditions']) == 9 and len(metrics['conditions']) == 9
    assert report['scoring'] == {'device': 'cpu', 'passes': 2}
    print(report['model_id'], report['run_id'], report['metrics_report_sha256'])
last = json.loads(REGISTRY.read_text(encoding='utf-8').splitlines()[-1])
assert last['status'] == 'completed' and last['evaluation_purpose'] == 'stage14_robustness'
print('Registry:', sha256(REGISTRY), 'lines:', len(REGISTRY.read_text(encoding='utf-8').splitlines()))
print('Stage 14 v1 completed. Do not commit per-clip score files; wait for human review.')


## What is committed after review

A later evidence PR may copy the registry, claim/environment/run reports, metrics reports and a hash manifest. Per-clip condition score files remain on Drive. No audio, feature arrays, model files, caches or secrets are committed. Results are reported exactly as produced.